# Tutorial: TrustSignal Vanta Evidence Master Index (2026-03-07)

Audience:
- TrustSignal engineering and security owners
- GRC reviewers preparing partner due-diligence packets
- Vanta integration and audit evidence reviewers

Purpose:
- Provide one notebook entry point for in-repo evidence mapping.
- Track control-to-artifact alignment and reproducible validation commands.
- Capture history context and open evidence gaps.


## Scope

This notebook intentionally references evidence already in the repository and does not claim external certifications.
Use it as a traceability index and export source for control evidence handoff.


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
import csv
import subprocess

ROOT = Path.cwd()
ROOT


## Evidence Inventory

The following paths are key evidence anchors used in current control mapping.


In [ ]:
evidence_inventory = [
    "README.md",
    "BLOCKED.md",
    "TASKS.md",
    "SECURITY_CHECKLIST.md",
    "docs/PRODUCTION_GOVERNANCE_TRACKER.md",
    "docs/final/10_INCIDENT_ESCALATION_AND_SLO_BASELINE.md",
    "docs/final/13_SOC2_READINESS_KICKOFF.md",
    "docs/final/14_VANTA_INTEGRATION_USE_CASE.md",
    "docs/ops/monitoring/alert-rules.yml",
    "docs/ops/monitoring/grafana-dashboard-deedshield-api.json",
    "docs/ops/monitoring/README.md",
    "docs/evidence/security/history-remediation-2026-02-25.md",
    "docs/evidence/security/github-org-hardening-2026-03-07.md",
    "docs/evidence/staging/vercel-staging-2026-02-27.md",
    "docs/evidence/staging/supabase-db-security-2026-02-27.md",
    "docs/evidence/staging/vanta-integration-2026-03-05-dry-run.md",
    "apps/api/src/security.ts",
    "apps/api/src/server.ts",
    "apps/api/src/services/registryAdapters.ts",
    "apps/api/src/registry-adapters.test.ts",
    "scripts/capture-staging-evidence.sh",
    "scripts/capture-vanta-integration-evidence.sh",
    "scripts/apply-github-branch-protection.sh",
    "scripts/capture-github-governance-evidence.sh",
    "notebooks/registry-wave1-primary-source-expansion-2026-03-07.ipynb",
]

[(path, (ROOT / path).exists()) for path in evidence_inventory]


## 90-Day History Context

Rolling timeline helps reviewers tie controls to development cadence and change management.


In [ ]:
since_date = (date.today() - timedelta(days=90)).isoformat()
history_cmd = [
    "git",
    "log",
    f"--since={since_date}",
    "--date=short",
    "--pretty=format:%h|%ad|%s",
]
history_lines = subprocess.run(history_cmd, capture_output=True, text=True, check=True).stdout.splitlines()
len(history_lines), history_lines[:25]


In [ ]:
area_keywords = {
    "registry": ["registry", "registries", "primary-source"],
    "security": ["security", "audit", "harden", "tls", "secret"],
    "vanta": ["vanta", "soc2", "compliance"],
    "api": ["api", "fastify", "route", "sdk"],
    "docs": ["docs", "readme", "governance", "plan"],
}

history_summary = {k: 0 for k in area_keywords}
history_summary["other"] = 0

for line in history_lines:
    lower = line.lower()
    matched = False
    for area, keywords in area_keywords.items():
        if any(keyword in lower for keyword in keywords):
            history_summary[area] += 1
            matched = True
            break
    if not matched:
        history_summary["other"] += 1

history_summary


## Vanta Control Matrix (Master)

Status values reflect in-repo implementation/evidence posture only (`READY`, `IN_PROGRESS`, `GAP`).


In [ ]:
vanta_controls_master = [
    {
        "control_id": "CC6.1",
        "control_family": "Access Control",
        "control_name": "API authentication and scoped authorization",
        "status": "READY",
        "objective": "Enforce API key authentication and scope checks across protected routes.",
        "evidence_files": [
            "apps/api/src/security.ts",
            "apps/api/src/server.ts",
        ],
        "validation_commands": [
            "npm -w apps/api run typecheck",
            "npm -w apps/api run test -- src/registry-adapters.test.ts",
        ],
    },
    {
        "control_id": "CC6.7",
        "control_family": "Change and Configuration",
        "control_name": "Production-safe startup and env guardrails",
        "status": "READY",
        "objective": "Block production boot when required trust-source credentials are missing.",
        "evidence_files": [
            "apps/api/src/server.ts",
            "README.md",
            "TASKS.md",
        ],
        "validation_commands": [
            "rg -n 'Missing required production env vars|TRUST_REGISTRY_SOURCE|PROPERTY_API_KEY' apps/api/src/server.ts",
        ],
    },
    {
        "control_id": "CC7.2",
        "control_family": "Change Management",
        "control_name": "Planned delivery tracking and evidence continuity",
        "status": "READY",
        "objective": "Track tasks, backlog sequencing, and linked evidence artifacts.",
        "evidence_files": [
            "TASKS.md",
            "docs/registry/free_primary_sources_catalog.md",
            "notebooks/vanta-evidence-master.ipynb",
        ],
        "validation_commands": [
            "git log --date=short --pretty=format:%h|%ad|%s --max-count=40",
            "git status --short",
        ],
    },
    {
        "control_id": "CC7.4",
        "control_family": "Governance and CI Assurance",
        "control_name": "Required checks and branch protection enforcement",
        "status": "IN_PROGRESS",
        "objective": "Restore CI required checks and enforce branch protection on master with evidence snapshots.",
        "evidence_files": [
            "BLOCKED.md",
            "TASKS.md",
            "docs/PRODUCTION_GOVERNANCE_TRACKER.md",
            "scripts/apply-github-branch-protection.sh",
            "scripts/capture-github-governance-evidence.sh",
        ],
        "validation_commands": [
            "./scripts/apply-github-branch-protection.sh <owner/repo> master",
            "./scripts/capture-github-governance-evidence.sh <owner/repo> master",
            "gh run list --limit 10",
        ],
    },
    {
        "control_id": "CC7.3",
        "control_family": "Security Operations",
        "control_name": "Fail-closed external dependency handling",
        "status": "READY",
        "objective": "Return compliance gaps when primary-source checks are unavailable.",
        "evidence_files": [
            "apps/api/src/services/registryAdapters.ts",
            "apps/api/src/registry-adapters.test.ts",
            "SECURITY_CHECKLIST.md",
        ],
        "validation_commands": [
            "npm -w apps/api run test -- src/registry-adapters.test.ts",
        ],
    },
    {
        "control_id": "CC8.1",
        "control_family": "Third-Party and Source Governance",
        "control_name": "Primary-source endpoint integrity",
        "status": "READY",
        "objective": "Limit registry adapters to official primary-source providers with host validation.",
        "evidence_files": [
            "apps/api/src/services/registryAdapters.ts",
            "docs/registry/free_primary_sources_catalog.md",
        ],
        "validation_commands": [
            "rg -n 'officialSourceName|primarySourceHost|validatePrimarySourceEndpoint' apps/api/src/services/registryAdapters.ts",
        ],
    },
    {
        "control_id": "CC9.2",
        "control_family": "Incident and Monitoring",
        "control_name": "Operational baseline and incident workflow",
        "status": "IN_PROGRESS",
        "objective": "Maintain incident process and SLO evidence while monitoring artifacts are deployed and fire/resolution evidence is captured.",
        "evidence_files": [
            "docs/final/10_INCIDENT_ESCALATION_AND_SLO_BASELINE.md",
            "docs/ops/monitoring/alert-rules.yml",
            "docs/ops/monitoring/grafana-dashboard-deedshield-api.json",
            "docs/ops/monitoring/README.md",
            "docs/PRODUCTION_GOVERNANCE_TRACKER.md",
            "TASKS.md",
        ],
        "validation_commands": [
            "promtool check rules docs/ops/monitoring/alert-rules.yml",
            "jq empty docs/ops/monitoring/grafana-dashboard-deedshield-api.json",
            "rg -n 'dashboard|alert|SLO|incident' docs/final/10_INCIDENT_ESCALATION_AND_SLO_BASELINE.md docs/PRODUCTION_GOVERNANCE_TRACKER.md TASKS.md",
        ],
    },
    {
        "control_id": "A1.2",
        "control_family": "Availability",
        "control_name": "Service health and status telemetry",
        "status": "READY",
        "objective": "Expose health/status/metrics endpoints for runtime inspection.",
        "evidence_files": [
            "apps/api/src/server.ts",
            "docs/evidence/staging/vercel-staging-2026-02-27.md",
        ],
        "validation_commands": [
            "rg -n '/api/v1/health|/api/v1/status|/api/v1/metrics' apps/api/src/server.ts",
        ],
    },
    {
        "control_id": "PI1.1",
        "control_family": "Processing Integrity",
        "control_name": "Vanta schema and normalized verification payload",
        "status": "READY",
        "objective": "Return deterministic, versioned evidence payloads suitable for partner ingestion.",
        "evidence_files": [
            "apps/api/src/server.ts",
            "docs/final/14_VANTA_INTEGRATION_USE_CASE.md",
            "docs/evidence/staging/vanta-integration-2026-03-05-dry-run.md",
        ],
        "validation_commands": [
            "rg -n '/api/v1/integrations/vanta/schema|/api/v1/integrations/vanta/verification' apps/api/src/server.ts",
        ],
    },
    {
        "control_id": "C1.1",
        "control_family": "Confidentiality",
        "control_name": "Secrets hygiene and historical remediation",
        "status": "IN_PROGRESS",
        "objective": "Maintain secret hygiene and complete external purge/rotation attestations.",
        "evidence_files": [
            "SECURITY_CHECKLIST.md",
            "docs/final/07_SECRET_ROTATION_AND_HISTORY_REMEDIATION.md",
            "docs/evidence/security/history-remediation-2026-02-25.md",
            "docs/evidence/security/github-org-hardening-2026-03-07.md",
            "scripts/capture-github-governance-evidence.sh",
            "TASKS.md",
        ],
        "validation_commands": [
            "rg -n 'Rotate|purge|history|secret' SECURITY_CHECKLIST.md TASKS.md docs/final/07_SECRET_ROTATION_AND_HISTORY_REMEDIATION.md",
            "./scripts/capture-github-governance-evidence.sh <owner/repo> master",
        ],
    },
]

vanta_controls_master


## Open Evidence Gaps

This extracts currently unchecked items from `TASKS.md` as a quick remediation queue.


In [ ]:
tasks_text = (ROOT / "TASKS.md").read_text(encoding="utf-8")
open_tasks = [line.strip() for line in tasks_text.splitlines() if line.strip().startswith("- [ ]")]
len(open_tasks), open_tasks[:20]


## Export Control Mapping

Exports an auditable CSV snapshot for handoff and tracker attachment.


In [ ]:
export_path = ROOT / "notebooks" / "artifacts" / "vanta-controls-master-2026-03-07.csv"
export_path.parent.mkdir(parents=True, exist_ok=True)

with export_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "control_id",
            "control_family",
            "control_name",
            "status",
            "objective",
            "evidence_files",
            "validation_commands",
        ],
    )
    writer.writeheader()
    for row in vanta_controls_master:
        writer.writerow({
            "control_id": row["control_id"],
            "control_family": row["control_family"],
            "control_name": row["control_name"],
            "status": row["status"],
            "objective": row["objective"],
            "evidence_files": " | ".join(row["evidence_files"]),
            "validation_commands": " | ".join(row["validation_commands"]),
        })

str(export_path)


## Reviewer Runbook

Run these commands before evidence handoff:

1. `npm -w apps/api run typecheck`
2. `npm -w apps/api run test -- src/registry-adapters.test.ts`
3. `git log --date=short --pretty=format:'%h|%ad|%s' --max-count=40`
4. `git status --short`

Then attach:
- `notebooks/vanta-evidence-master.ipynb`
- `notebooks/artifacts/vanta-controls-master-2026-03-07.csv`
- `notebooks/registry-wave1-primary-source-expansion-2026-03-07.ipynb`


## Session Update - 2026-03-07 CI/Governance Unblock

- Added dedicated remediation notebook: `notebooks/governance-ci-unblock-2026-03-07.ipynb`.
- Captured session control CSV: `notebooks/artifacts/vanta-controls-ci-unblock-2026-03-07.csv`.
- Remediation files: `.github/workflows/ci.yml`, `src/verifiers/zkmlVerifier.ts`.
- Local validation: `npx tsc --strict --noEmit` and `npx vitest run --coverage` passed.
- Remote gate remains blocked pending push and GitHub rerun confirmation on `master`.


In [ ]:
session_artifacts = [
    'notebooks/governance-ci-unblock-2026-03-07.ipynb',
    'notebooks/artifacts/vanta-controls-ci-unblock-2026-03-07.csv',
    '.github/workflows/ci.yml',
    'src/verifiers/zkmlVerifier.ts',
]
session_artifacts
